# HW04 序列模型

## 2.1 理论计算题

**题目：** 给定序列 `"ababc"`，使用一阶马尔可夫模型 $p(x_t \mid x_{t-1})$ 和拉普拉斯平滑，词表为 $\{'a', 'b', 'c'\}$，估计 $p('a' \mid 'b')$ 和 $p('c' \mid 'b')$。计算时需考虑所有可能的转移（包括序列中未出现的转移）。

### 拉普拉斯平滑公式

$$
p(x_t \mid x_{t-1}) = \frac{C(x_{t-1}, x_t) + 1}{C(x_{t-1}) + |V|}
$$

其中：
- $C(x_{t-1}, x_t)$：转移 $(x_{t-1}, x_t)$ 在序列中的出现次数
- $C(x_{t-1})$：字符 $x_{t-1}$ 作为前缀出现的总次数
- $|V| = 3$：词表大小

### 从序列 `"ababc"` 提取转移

序列分解：a → b → a → b → c，共 4 个转移（二元组）。

| 转移 | 出现次数 |
|------|----------|
| a → b | 2 |
| b → a | 1 |
| b → c | 1 |

前缀统计：

| 前缀字符 | 作为前缀出现次数 $C(x_{t-1})$ |
|----------|-------------------------------|
| a | 2 |
| b | 2 |

以 `'b'` 为前缀：$C(b) = 2$，$C(b, a) = 1$，$C(b, c) = 1$。

### 计算过程

**$p('a' \mid 'b')$：**

$$
p('a' \mid 'b') = \frac{C(b, a) + 1}{C(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$

**$p('c' \mid 'b')$：**

$$
p('c' \mid 'b') = \frac{C(b, c) + 1}{C(b) + |V|} = \frac{1 + 1}{2 + 3} = \frac{2}{5} = 0.4
$$

### 最终答案

- $p('a' \mid 'b') = \dfrac{2}{5} = 0.4$
- $p('c' \mid 'b') = \dfrac{2}{5} = 0.4$

In [ ]:
2.2 编程题

编写函数 `preprocess_text(text, n)`，完成文本预处理并生成自回归语言模型的训练样本。

In [1]:
import re
from collections import Counter


def preprocess_text(text, n):
    """
    文本预处理：清洗、分词、构建词表、生成滑动窗口特征与标签。

    Parameters
    ----------
    text : str
        输入文本
    n : int
        特征序列长度（上下文窗口大小）

    Returns
    -------
    vocab : dict
        词表，键为单词，值为从 0 开始的整数 ID（按出现频率降序，同频按首次出现顺序）
    features : list
        长度为 n 的特征序列列表
    labels : list
        每个特征对应的下一个词标签
    """
    # 1. 转小写，仅保留字母和空格
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)

    # 2. 按空格分词
    words = text.split()

    # 3. 构建词表：按出现频率排序，同频按首次出现顺序，ID 从 0 开始
    freq = Counter(words)
    first_idx = {}
    for i, word in enumerate(words):
        if word not in first_idx:
            first_idx[word] = i
    sorted_words = sorted(freq.keys(), key=lambda w: (-freq[w], first_idx[w]))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    # 4. 滑动窗口生成特征与标签（无后续词则忽略）
    features = []
    labels = []
    for i in range(len(words) - n):
        features.append(words[i:i + n])
        labels.append(words[i + n])

    return vocab, (features, labels)


# 示例测试
vocab, (features, labels) = preprocess_text("The time machine", 2)
print("词表:", vocab)
print("特征:", features)
print("标签:", labels)

词表: {'the': 0, 'time': 1, 'machine': 2}
特征: [['the', 'time']]
标签: ['machine']


# 3 循环神经网络

## 3.1 理论计算题

**模型：**
- 隐藏状态：$h_t = W_{hh} h_{t-1} + W_{hx} x_t$（线性，无偏置）
- 输出：$o_t = W_{oh} h_t$
- 损失：$L = \dfrac{1}{2} \displaystyle\sum_{t=1}^{T} (o_t - y_t)^2$

### （1）BPTT 推导 $\dfrac{\partial L}{\partial W_{hh}}$

**前向传播：**

$$
h_t = W_{hh} h_{t-1} + W_{hx} x_t, \quad o_t = W_{oh} h_t
$$

**单步损失对输出的梯度：**

$$
\frac{\partial L}{\partial o_t} = o_t - y_t
$$

**BPTT 反向传播隐藏状态误差（从 $t = T$ 到 $1$）：**

$$
\delta h_t = \frac{\partial L}{\partial h_t} = W_{oh}^\top (o_t - y_t) + W_{hh}^\top \delta h_{t+1}
$$

其中 $\delta h_{T+1} = 0$。

**对 $W_{hh}$ 的梯度：**

由于 $h_t = W_{hh} h_{t-1} + W_{hx} x_t$，有 $\dfrac{\partial h_t}{\partial W_{hh}} = h_{t-1}^\top$（外积形式），因此：

$$
\boxed{\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \delta h_t \, h_{t-1}^\top}
$$

**展开到所有时间步（显示时间步之间的链式传播）：**

$$
\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^{T} \sum_{k=t}^{T} \left( \prod_{j=t+1}^{k} W_{hh}^\top \right) W_{oh}^\top (o_k - y_k) \, h_{t-1}^\top
$$

其中当 $k = t$ 时，空乘积 $\displaystyle\prod_{j=t+1}^{t} = I$（单位矩阵）。

含义：时刻 $t$ 对 $W_{hh}$ 的梯度，不仅受当前时刻 $t$ 的损失影响，还受之后所有时刻 $k > t$ 的损失影响，中间通过 $W_{hh}^\top$ 连乘传播。

### （2）梯度消失与梯度爆炸的条件

在 BPTT 中，$\delta h_t$ 包含形如 $\displaystyle\prod_{j=t+1}^{k} W_{hh}^\top$ 的连乘项，序列越长，连乘次数越多。

| 情况 | 条件 | 现象 |
|------|------|------|
| **梯度消失** | $W_{hh}$ 的最大奇异值（或谱半径）$\rho(W_{hh}) < 1$ | 连乘趋近于 0，远距离时间步的梯度几乎无法传到早期时刻，难以学习长期依赖 |
| **梯度爆炸** | $\rho(W_{hh}) > 1$ | 连乘指数增长，梯度数值过大，训练不稳定 |
| **梯度稳定** | $\rho(W_{hh}) \approx 1$ | 梯度可维持，但线性 RNN 在此条件下仍难以有效建模长期依赖 |

实际中常采用梯度裁剪（防爆炸）、LSTM/GRU 门控机制（缓解消失）等方法。

## 3.2 编程题

实现简单 RNN 单元的前向传播与单步反向传播（仅计算梯度，不更新参数）。

- 前向：$h_t = \tanh(W_{hx} x_t + W_{hh} h_{t-1} + b_h)$
- 反向：给定上游梯度 `dh_next`（即 $\partial L / \partial h_t$），计算 `dx_t`, `dh_prev`, `dW_hx`, `dW_hh`, `db_h`

In [2]:
import numpy as np


def rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN 单步前向传播。

    Parameters
    ----------
    x_t : ndarray, shape (batch_size, input_size)
    h_prev : ndarray, shape (batch_size, hidden_size)
    W_hx : ndarray, shape (input_size, hidden_size)
    W_hh : ndarray, shape (hidden_size, hidden_size)
    b_h : ndarray, shape (hidden_size,)

    Returns
    -------
    h_t : ndarray, shape (batch_size, hidden_size)
    cache : tuple, 用于反向传播的中间变量
    """
    z = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(z)
    cache = (x_t, h_prev, W_hx, W_hh, z, h_t)
    return h_t, cache


def rnn_step_backward(dh_next, cache):
    """
    RNN 单步反向传播。

    Parameters
    ----------
    dh_next : ndarray, shape (batch_size, hidden_size)
        损失对当前隐藏状态 h_t 的梯度
    cache : tuple
        前向传播保存的中间变量

    Returns
    -------
    dx_t, dh_prev, dW_hx, dW_hh, db_h
    """
    x_t, h_prev, W_hx, W_hh, z, h_t = cache

    # tanh 导数: 1 - tanh(z)^2
    dz = dh_next * (1.0 - h_t ** 2)

    dW_hx = x_t.T @ dz
    dW_hh = h_prev.T @ dz
    db_h = np.sum(dz, axis=0)
    dx_t = dz @ W_hx.T
    dh_prev = dz @ W_hh.T

    return dx_t, dh_prev, dW_hx, dW_hh, db_h


# 简单测试
np.random.seed(42)
batch_size, input_size, hidden_size = 2, 3, 4

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
b_h = np.zeros(hidden_size)

h_t, cache = rnn_step_forward(x_t, h_prev, W_hx, W_hh, b_h)
dh_next = np.random.randn(batch_size, hidden_size)

dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_step_backward(dh_next, cache)

print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)


# 4 高级循环神经网络

## 4.1 理论计算题

**题目：** 深度双向 RNN，共 $L$ 层，每层隐藏单元数 $H$，输入维度 $D$，输出维度 $O$（仅最后输出层）。计算参数总数（含所有全连接层权重与偏置），忽略嵌入层及输出层前的投影。

### 单层单向 RNN 参数量

输入维度 $I$，隐藏维度 $H$：

$$
\text{Params}_{\text{uni}} = \underbrace{I \cdot H}_{W_{ih}} + \underbrace{H \cdot H}_{W_{hh}} + \underbrace{H}_{b_h} = IH + H^2 + H
$$

### 各层参数分解

**第 1 层（输入维度 $D$，双向）：**

$$
\text{Params}_1 = 2(DH + H^2 + H) = 2DH + 2H^2 + 2H
$$

**第 $2, \ldots, L$ 层（输入为上一层拼接后的 $2H$，双向）：**

每层每个方向：$2H \cdot H + H^2 + H = 3H^2 + H$

$$
\text{Params}_{l} = 2(3H^2 + H) = 6H^2 + 2H, \quad l = 2, \ldots, L
$$

**输出层（$2H \to O$）：**

$$
\text{Params}_{\text{out}} = 2H \cdot O + O = 2HO + O
$$

### 参数总数

$$
\boxed{
\text{Total} = 2DH + 2H^2 + 2H + (L-1)(6H^2 + 2H) + 2HO + O
}
$$

**化简形式：**

$$
\text{Total} = 2H(D + O + L) + O + 2H^2(3L - 2)
$$

### 参数明细表

| 模块 | 参数量 |
|------|--------|
| 第 1 层 BiRNN | $2DH + 2H^2 + 2H$ |
| 第 $2 \sim L$ 层 BiRNN（共 $L-1$ 层）| $(L-1)(6H^2 + 2H)$ |
| 输出全连接层 | $2HO + O$ |
| **合计** | $2H(D+O+L) + O + 2H^2(3L-2)$ |

## 4.2 编程题

实现双向 RNN 编码器，输入序列 $X$（形状 `(seq_len, batch, input_dim)`），返回：

1. 每个时间步拼接后的前向/后向隐藏状态，形状 `(seq_len, batch, 2*hidden_dim)`
2. 序列最终表示：前向与后向**最终隐藏状态**的拼接，形状 `(batch, 2*hidden_dim)`

In [1]:
import torch
import torch.nn as nn


class BiRNNEncoder(nn.Module):
    """双向 RNN 编码器"""

    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=False,
            bidirectional=True,
        )

    def forward(self, X):
        """
        Parameters
        ----------
        X : Tensor, shape (seq_len, batch, input_dim)

        Returns
        -------
        outputs : Tensor, shape (seq_len, batch, 2 * hidden_dim)
            每个时间步拼接后的前向/后向隐藏状态
        seq_repr : Tensor, shape (batch, 2 * hidden_dim)
            序列表示（两方向最终隐藏状态的拼接）
        """
        outputs, h_n = self.rnn(X)
        # h_n: (num_layers * 2, batch, hidden_dim)
        # 最后一层：h_n[-2] 为前向最终状态，h_n[-1] 为后向最终状态
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        seq_repr = torch.cat([h_forward, h_backward], dim=-1)
        return outputs, seq_repr


# 测试
torch.manual_seed(42)
seq_len, batch_size, input_dim, hidden_dim = 5, 3, 4, 8

encoder = BiRNNEncoder(input_dim, hidden_dim)
X = torch.randn(seq_len, batch_size, input_dim)

outputs, seq_repr = encoder(X)
print("outputs shape:", outputs.shape)      # (5, 3, 16)
print("seq_repr shape:", seq_repr.shape)    # (3, 16)

outputs shape: torch.Size([5, 3, 16])
seq_repr shape: torch.Size([3, 16])


# 5 嵌入向量

## 5.1 理论计算题

**题目：** 给定中心词 $w_c$ 和上下文词 $w_o$，用负采样（$K$ 个负样本）推导 Skip-gram 的损失函数，并说明负样本的采样方式。词向量记为 $\mathbf{v}_c$、$\mathbf{u}_o$，负样本词向量记为 $\mathbf{u}_{n_k}$。

### Skip-gram 负采样目标函数

Skip-gram 将多分类问题转化为二分类：区分"中心词与上下文词共现"（正样本）和"中心词与噪声词共现"（负样本）。

对一对 $(w_c, w_o)$，定义：

- **正样本**：$(w_c, w_o)$ 应使 $\sigma(\mathbf{u}_o^\top \mathbf{v}_c)$ 接近 1
- **负样本**：$(w_c, n_k)$ 应使 $\sigma(\mathbf{u}_{n_k}^\top \mathbf{v}_c)$ 接近 0，即 $\sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_c)$ 接近 1

其中 $\sigma(x) = \dfrac{1}{1 + e^{-x}}$ 为 sigmoid 函数。

**对数似然（需最大化）：**

$$
\mathcal{L} = \log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) + \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_c)
$$

**损失函数（负对数似然，需最小化）：**

$$
\boxed{
\mathcal{J} = -\log \sigma(\mathbf{u}_o^\top \mathbf{v}_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_c)
}
$$

对整个训练集，总目标为所有 $(w_c, w_o)$ 对的 $\mathcal{J}$ 之和（或均值）。

### 负样本采样方式

1. **定义噪声分布** $P_n(w)$：通常根据词频采样，高频词更易被选为负样本。Word2Vec 中使用：
   $$P_n(w_i) = \frac{f(w_i)^{3/4}}{\displaystyle\sum_{j=1}^{V} f(w_j)^{3/4}}$$
   其中 $f(w_i)$ 为词 $w_i$ 在语料中的频率。指数 $3/4$ 可缓解极端高频词的主导作用。

2. **采样过程**：对每个正样本 $(w_c, w_o)$，从 $P_n(w)$ 中独立抽取 $K$ 个词作为负样本 $\{n_1, n_2, \ldots, n_K\}$，通常排除正样本上下文词 $w_o$。

3. **直观理解**：负样本应代表"与中心词不共现"的随机词，用词频分布近似真实语言中的词出现概率，使模型学会区分真实上下文与随机噪声。

## 5.2 编程题

实现 CBOW 模型的前向传播与损失计算（完整 softmax，不使用负采样）。

- 输入：一批上下文词索引、词表大小 $V$、嵌入维度 $d$、输入权重 $W \in \mathbb{R}^{V \times d}$、输出权重 $W_{out} \in \mathbb{R}^{d \times V}$
- 步骤：上下文词向量取平均 → 隐藏层 → softmax 输出分布 → 与中心词计算交叉熵损失

In [2]:
import torch
import torch.nn.functional as F


def cbow_loss(context_indices, center_indices, W, W_out):
    """
    CBOW 前向传播并计算交叉熵损失。

    Parameters
    ----------
    context_indices : Tensor or list, shape (batch_size, context_size)
        每个样本的上下文词索引
    center_indices : Tensor or list, shape (batch_size,)
        每个样本的中心词索引（预测目标）
    W : Tensor, shape (V, d)
        输入词嵌入矩阵
    W_out : Tensor, shape (d, V)
        输出权重矩阵

    Returns
    -------
    loss : Tensor (scalar)
        交叉熵损失
    """
    if not isinstance(context_indices, torch.Tensor):
        context_indices = torch.tensor(context_indices, dtype=torch.long)
    if not isinstance(center_indices, torch.Tensor):
        center_indices = torch.tensor(center_indices, dtype=torch.long)

    # 1. 查表得到上下文词向量，取平均作为隐藏层
    context_emb = W[context_indices]          # (batch, context_size, d)
    h = context_emb.mean(dim=1)               # (batch, d)

    # 2. 计算输出 logits 与概率分布（softmax 在 cross_entropy 内部完成）
    logits = h @ W_out                        # (batch, V)

    # 3. 交叉熵损失，目标为中心词索引
    loss = F.cross_entropy(logits, center_indices)

    return loss


# 测试
torch.manual_seed(42)
V, d, context_size, batch_size = 100, 50, 4, 8

W = torch.randn(V, d) * 0.1
W_out = torch.randn(d, V) * 0.1

context_indices = torch.randint(0, V, (batch_size, context_size))
center_indices = torch.randint(0, V, (batch_size,))

loss = cbow_loss(context_indices, center_indices, W, W_out)
print("CBOW loss:", loss.item())
print("logits shape:", (torch.randn(batch_size, d) @ W_out).shape)

CBOW loss: 4.590110778808594
logits shape: torch.Size([8, 100])


# 6 注意力机制

## 6.1 理论计算题

**题目：** 给定 $Q \in \mathbb{R}^{2 \times 4}$，$K \in \mathbb{R}^{3 \times 4}$，$V \in \mathbb{R}^{3 \times 5}$，计算缩放点积注意力（无 mask）的输出，$d_k = 4$。

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

### 步骤一：计算得分矩阵

$$S = \frac{QK^\top}{\sqrt{d_k}} = \frac{QK^\top}{2} \in \mathbb{R}^{2 \times 3}$$

$S_{ij}$ 表示第 $i$ 个 Query 与第 $j$ 个 Key 的相似度。

**示例**（取 $Q = \begin{bmatrix}1&1&0&0\\0&0&1&1\end{bmatrix}$，$K = \begin{bmatrix}1&0&0&0\\0&1&0&0\\0&0&1&0\end{bmatrix}$）：

$$QK^\top = \begin{bmatrix}1&1&0\\0&0&1\end{bmatrix}, \quad S = \begin{bmatrix}0.5&0.5&0\\0&0&0.5\end{bmatrix}$$

### 步骤二：对每行做 Softmax

对 $S$ 的每一行独立做 softmax，得到注意力权重矩阵 $A \in \mathbb{R}^{2 \times 3}$：

$$A_{ij} = \frac{\exp(S_{ij})}{\displaystyle\sum_{k=1}^{3} \exp(S_{ik})}$$

**示例结果：**

$$A \approx \begin{bmatrix}0.3837&0.3837&0.2327\\0.2741&0.2741&0.4519\end{bmatrix}$$

每行和为 1，表示每个 Query 对 3 个 Key 的注意力分布。

### 步骤三：加权求和得到输出

$$\text{Output} = AV \in \mathbb{R}^{2 \times 5}$$

第 $i$ 行输出为 Value 矩阵各行按 $A_{i,:}$ 的加权平均。

**示例**（取 $V = \begin{bmatrix}1&2&3&4&5\\6&7&8&9&10\\11&12&13&14&15\end{bmatrix}$）：

$$\text{Output} \approx \begin{bmatrix}5.25&6.25&7.25&8.25&9.25\\6.89&7.89&8.89&9.89&10.89\end{bmatrix}$$

### 维度总结

| 步骤 | 公式 | 输出形状 |
|------|------|----------|
| 得分 | $S = QK^\top / \sqrt{d_k}$ | $(2, 3)$ |
| 权重 | $A = \text{softmax}(S)$ | $(2, 3)$ |
| 输出 | $\text{Output} = AV$ | $(2, 5)$ |

## 6.2 编程题

实现多头注意力（Multi-Head Attention）前向传播：`num_heads = 2`，`d_model = 4`，输入 `X` 形状 `(seq_len, batch, d_model)`，输出形状与输入相同。

In [4]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    """多头注意力前向传播"""

    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # d_k = d_v = 2

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, X):
        """
        Parameters
        ----------
        X : Tensor, shape (seq_len, batch, d_model)

        Returns
        -------
        output : Tensor, shape (seq_len, batch, d_model)
        """
        seq_len, batch, _ = X.shape

        # 1. 线性投影得到 Q, K, V
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # 2. 拆分为多个头: (seq_len, batch, num_heads, d_k) -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)

        # 3. 每个头做缩放点积注意力
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attn = F.softmax(scores, dim=-1)
        head_out = attn @ V  # (batch, num_heads, seq_len, d_k)

        # 4. 拼接所有头: (seq_len, batch, d_model)
        head_out = head_out.permute(2, 0, 1, 3).contiguous()
        concat = head_out.view(seq_len, batch, self.d_model)

        # 5. 最终线性层
        output = self.W_o(concat)
        return output


# 测试
torch.manual_seed(42)
seq_len, batch_size, d_model, num_heads = 5, 3, 4, 2

mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
X = torch.randn(seq_len, batch_size, d_model)
output = mha(X)

print("input shape:", X.shape)
print("output shape:", output.shape)
print("d_k per head:", mha.d_k)

input shape: torch.Size([5, 3, 4])
output shape: torch.Size([5, 3, 4])
d_k per head: 2
